In [0]:
fact_incidents = spark.sql(f"select * from regis_healthcare.silver.incidents;")
fact_incidents.createOrReplaceTempView("incidents")

In [0]:
# Fact_Incidents -- > Source: incidents
# | Foreign Keys      |
# | ----------------- |
# | incident_key      |
# | resident_key      |
# | facility_key      |
# | employee_key      |
# | incident_type_key |
# | incident_date_key |

from pyspark.sql.functions import col, date_format
# Create date_key column in YYYYMMDD format
fact_incidents = fact_incidents.withColumn("incident_date_key", date_format(col("incident_date"), "yyyyMMdd"))
# Optionally cast to integer for warehouse-style keys
fact_incidents = fact_incidents.withColumn("incident_date_key", col("incident_date_key").cast("int"))
#-------------------------
from pyspark.sql.functions import col, regexp_replace
fact_incidents = fact_incidents.withColumn(
    "resident_key",
    regexp_replace(col("resident_id"), "^RES", "").cast("int")
)
fact_incidents = fact_incidents.withColumn(
    "incident_key",
    regexp_replace(col("incident_id"), "^INC", "").cast("int")
)

fact_incidents = fact_incidents.withColumn(
    "facility_key",
    regexp_replace(col("facility_id"), "^FAC", "").cast("int")
)

fact_incidents = fact_incidents.withColumn(
    "employee_key",
    regexp_replace(col("employee_id"), "^EMP", "").cast("int")
)
from pyspark.sql.functions import col,when

fact_incidents = fact_incidents.withColumn("incident_type_key",when(col("incident_type")=="AGGRESSION",1)
     .when(col("incident_type")=="BEHAVIOURAL ISSUE",2)
     .when(col("incident_type")=="CHOKING",3)
     .when(col("incident_type")=="ELOPEMENT",4)
     .when(col("incident_type")=="EQUIPMENT FAILURE",5)
     .when(col("incident_type")=="FALL",6)
     .when(col("incident_type")=="INFECTION",7)
     .when(col("incident_type")=="MEDICATION ERROR",8)
     .when(col("incident_type")=="OTHER",9)
     .when(col("incident_type")=="PRESSURE INJURY",10)
     .when(col("incident_type")=="SKIN TEAR",11)
     .when(col("incident_type")=="UNKNOWN",12)
     .when(col("incident_type")=="WANDERING",13).otherwise(0).cast("int"))

fact_incidents = fact_incidents.select(
 "incident_key",     
 "resident_key",    
 "facility_key",   
 "employee_key", 
 "incident_type_key", 
 "incident_date_key" 
)

display(fact_incidents)


#### cataloge 

In [0]:
fact_incidents.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.fact_incidents")

In [0]:
fact_incidents.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.saveAsTable(f"regis_healthcare.gold.sb_fact_incidents")
print(fact_incidents.count())

In [0]:
from delta.tables import DeltaTable
from pyspark.sql import functions as F

# Load Delta table with correct fully-qualified name
delta_table = DeltaTable.forName(spark, "regis_healthcare.gold.fact_incidents")
# Create DataFrame from source table with correct fully-qualified name
# sb_dim_products
df_child_products = (
    spark.table("regis_healthcare.gold.sb_fact_incidents")
    .select("*")
)
# Perform merge
delta_table.alias("target").merge(
    source=df_child_products.alias("source"),
    condition="target.incident_key = source.incident_key"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
dim_df = spark.sql(f"select * from regis_healthcare.gold.fact_incidents;")
print(dim_df.count())

sb_dim_df = spark.sql(f"select * from regis_healthcare.gold.sb_fact_incidents;")
print(sb_dim_df.count())

#### s3 loading

In [0]:
# gold load to s3
fact_incidents.write\
    .format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
        .option("delta.enableChangeDataFeed","true")\
            .mode("overwrite")\
.save(f"s3://regis-healthcare/gold-delta-table/fact_incidents")

In [0]:
from delta.tables import DeltaTable

# ✅ Path to your Delta table stored in S3
delta_table_path = f"s3://regis-healthcare/gold-delta-table/fact_incidents"

# ✅ Load target Delta table
delta_table = DeltaTable.forPath(spark, delta_table_path)

# ✅ Source DataFrame (example: df_child_products)
source_df = fact_incidents

# ✅ Perform MERGE with upsert logic
(
    delta_table.alias("target")
    .merge(
        source_df.alias("source"),
        "target.incident_key = source.incident_key"
    )
    .whenMatchedUpdateAll()      # Update all columns when matched
    .whenNotMatchedInsertAll()   # Insert all columns when not matched
    .execute()
)
